[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/090.%20The%20Global%20Sourcing%20Offshoring%20vs.%20Reshoring%20Problem/P90-Tier-5_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 90. The Global Sourcing Offshoring vs. Reshoring Problem
## Tier 5: An Integrated Digital-Twin Simulation

### Purpose and scope

This notebook builds a small, reproducible simulation of a global sourcing system. It connects supplier operations, logistics status, market demand, scenario analysis, a score-based sourcing allocation rule, and key performance indicators (KPIs). The goal is to show how these components can interact in a digital-twin-style workflow and how a what-if scenario can change the model's outputs.

The model is an educational simulation, not a validated digital twin of a real company. Its sensor readings, demand changes, disruptions, costs, and scenario effects are generated or simplified by explicit rules in the code. They should not be interpreted as live observations, forecasts, or verified business results.

### Inputs, process, and outputs

- **Inputs:** supplier capacities, unit costs, risk factors, quality scores, product demand, scenario parameters, and the random seed.
- **Process:** generate simulated system observations, calculate supplier scores, allocate each product in proportion to those scores, calculate KPIs, and evaluate selected what-if scenarios.
- **Outputs:** allocation dictionaries, scenario results, KPI summaries, and visualizations of values generated by the simulation.

### Model assumptions and limitations

- The allocation rule is a score-based heuristic. It does not solve a constrained optimization problem.
- The same supplier set is considered for every product. Product-specific eligibility is not modeled.
- Allocation is proportional to supplier scores; supplier capacity limits are reported as input data but are not enforced as allocation constraints. Consequently, an allocation can exceed a supplier's capacity.
- Unit costs are treated as comparable across products, and total procurement cost is calculated as allocated quantity multiplied by unit cost. There is no additional fixed-cost term.
- Risk exposure is the sum of allocated quantity multiplied by the supplier's risk factor. The quality KPI is a quantity-weighted sum of supplier quality scores, not a percentage or a normalized index.
- Logistics and market components generate simulated status data, but the allocation rule does not use logistics costs, lead times, or product-specific supplier suitability.
- Scenario calculations are simplified illustrations. In particular, the currency scenario uses a stated hedging-effectiveness assumption, while the port-closure scenario uses direct arithmetic rather than a route-level logistics network model.

### Allocation rule

For each product, the model assigns a share of demand to every supplier in proportion to a weighted score. Lower cost and lower risk increase a supplier's score; higher quality also increases it. The rule is intentionally kept visible so that its behavior can be inspected.

For supplier $i$, define

$$s_i = 0.6\left(\frac{1}{c_i}\right) + 0.3\left(\frac{1}{1+r_i}\right) + 0.1q_i,$$

where $c_i$ is the supplier's current unit cost, $r_i$ is its adjusted risk factor, and $q_i$ is its current quality score. For product $p$ with demand $d_p$, the allocation to supplier $i$ is

$$x_{pi} = d_p\frac{s_i}{\sum_j s_j}.$$

Here, $x_{pi}$ is the quantity of product $p$ assigned to supplier $i$. The score weights are the model's stated heuristic; the terms have different scales, so the weights do not by themselves guarantee a particular trade-off in monetary or physical units. The code implements this rule in `GlobalSourcingDigitalTwin.optimize_sourcing_allocation`.

### Simulation sequence

1. Define supplier, logistics, and market components.
2. Connect the components in an integrated sourcing model.
3. Calculate an initial score-based allocation and its KPIs.
4. Evaluate tariff, port-closure, and currency scenarios.
5. Update the simulated system over time, including occasional supplier disruptions.
6. Plot and interpret the outputs produced by this simulation.

The fixed random seed makes the random components reproducible when the notebook is run from top to bottom in the same software environment. Simulation speed is a model setting; it does not mean that the Python code runs faster than real time.

In [ ]:
from copy import deepcopy
from datetime import datetime, timedelta
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Keep simulated sensor readings and disruption events reproducible.
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Define the digital-twin components

A **digital twin** is a computational representation of a system that is updated using information about that system. Here, the representation is deliberately small: supplier objects hold operating and sensor data, a logistics object tracks simulated port congestion, and a market object tracks simulated product demand.

These components provide state for the later simulation. A **state** is the collection of values that describe the modeled system at a particular time. Updating state is necessary because scenario analysis and KPI calculations need a defined system condition to evaluate. In this notebook, state changes through explicit simulation rules; the model is not connected to physical sensors or live data.

In [ ]:
class SupplierDigitalTwin:
    """Represent one supplier and its simulated operational observations."""

    def __init__(self, name, location, capacity, cost, risk, quality):
        self.name = name
        self.location = location
        self.base_capacity = capacity
        self.current_capacity = capacity
        self.base_cost = cost
        self.current_cost = cost
        self.risk_factor = risk
        self.quality_score = quality

        self.operational_status = 1.0
        self.production_efficiency = 1.0
        self.quality_control_score = 1.0

        self.sensor_data = {
            "production_rate": [],
            "quality_metrics": [],
            "equipment_status": [],
            "inventory_levels": [],
            "delivery_performance": [],
            "operational_uptime": [],
        }

        self.disruption_probability = 0.0
        self.capacity_forecast = []
        self.cost_trend = []

    def update_sensor_data(self, timestamp):
        """Append simulated sensor observations for one time step."""
        production_rate = (
            self.current_capacity
            * self.operational_status
            * np.random.normal(1.0, 0.05)
        )
        quality_metric = (
            self.quality_score
            * self.quality_control_score
            * np.random.normal(1.0, 0.02)
        )
        equipment_status = (
            self.operational_status * np.random.normal(1.0, 0.03)
        )
        inventory_level = np.random.uniform(0.3, 0.8) * self.current_capacity
        delivery_performance = (
            np.random.uniform(0.85, 0.98) * self.operational_status
        )
        operational_uptime = np.random.uniform(0.92, 0.99)

        observations = {
            "production_rate": production_rate,
            "quality_metrics": quality_metric,
            "equipment_status": equipment_status,
            "inventory_levels": inventory_level,
            "delivery_performance": delivery_performance,
            "operational_uptime": operational_uptime,
        }
        for metric, value in observations.items():
            self.sensor_data[metric].append((timestamp, value))

    def apply_disruption(self, disruption_type, severity):
        """Change a supplier parameter according to a specified disruption."""
        if disruption_type == "tariff_increase":
            self.current_cost = self.base_cost * (1 + severity)
        elif disruption_type == "capacity_reduction":
            self.current_capacity = self.base_capacity * (1 - severity)
        elif disruption_type == "quality_issue":
            self.quality_control_score = 1.0 - severity
        elif disruption_type == "operational_issue":
            self.operational_status = 1.0 - severity
        else:
            raise ValueError(f"Unknown disruption type: {disruption_type}")

        self.disruption_probability = severity


class LogisticsDigitalTwin:
    """Represent logistics locations and simulated port congestion."""

    def __init__(self):
        self.routes = {}
        self.ports = {}
        self.transportation_costs = {}
        self.lead_times = {}
        self.shipment_tracking = {}
        self.port_congestion = {}
        self.weather_impacts = {}
        self.congestion_forecasts = {}
        self.delay_predictions = {}

    def update_network_status(self, timestamp):
        """Generate and store a congestion observation for each port."""
        for port in self.ports:
            congestion = np.random.beta(2, 5)
            self.port_congestion[port] = congestion
            self.congestion_forecasts.setdefault(port, []).append(
                (timestamp, congestion)
            )


class MarketDemandTwin:
    """Represent product demand and its simulated changes over time."""

    def __init__(self):
        self.demand_forecasts = {}
        self.market_conditions = {}
        self.competitor_actions = {}
        self.customer_preferences = {}
        self.current_demand = {}
        self.price_elasticity = {}
        self.market_trends = {}

    def update_market_data(self, timestamp):
        """Apply a small random demand change and record the resulting value."""
        for product in self.current_demand:
            demand_change = np.random.normal(0, 0.05)
            self.current_demand[product] *= 1 + demand_change
            self.demand_forecasts.setdefault(product, []).append(
                (timestamp, self.current_demand[product])
            )


print("Supplier, logistics, and market components are defined.")

## 2. Integrate the system and calculate sourcing allocations

The integrated model stores the supplier, logistics, and market components and provides methods to update their states, calculate an allocation, and report KPIs. The allocation method follows the score-based rule defined above: it computes a score for each supplier, normalizes the scores so their shares sum to one, and applies those shares to each product's demand.

This normalization ensures that the allocated quantities for a product sum to that product's demand, apart from floating-point rounding. It does **not** ensure that any supplier's capacity is respected. That distinction matters when interpreting the result: the output is a heuristic allocation, not a capacity-feasible production plan.

The risk KPI measures allocated quantity multiplied by risk factor. The quality KPI measures allocated quantity multiplied by quality score. These are aggregate exposure and quality-weighted quantity measures, respectively; neither is a probability or a percentage.

In [ ]:
class GlobalSourcingDigitalTwin:
    """Integrate supplier, logistics, and market simulation components."""

    def __init__(self):
        self.suppliers = {}
        self.logistics = LogisticsDigitalTwin()
        self.market = MarketDemandTwin()

        self.current_time = datetime(2024, 1, 1)
        self.simulation_speed = 1000

        self.current_allocation = {}
        self.optimization_results = {}
        self.kpi_history = {
            "total_cost": [],
            "supply_risk": [],
            "quality_score": [],
            "delivery_performance": [],
            "inventory_levels": [],
            "disruption_events": [],
        }

        self.scenario_engine = None
        self.current_scenarios = []
        self.risk_predictions = {}
        self.demand_predictions = {}
        self.cost_projections = {}

    def initialize_suppliers(self):
        """Create the example suppliers, ports, and product demands."""
        self.suppliers["China_Electronics"] = SupplierDigitalTwin(
            "China_Electronics", "China", 50000, 25, 0.8, 0.7
        )
        self.suppliers["China_Manufacturing"] = SupplierDigitalTwin(
            "China_Manufacturing", "China", 45000, 35, 0.8, 0.75
        )
        self.suppliers["Mexico_Auto"] = SupplierDigitalTwin(
            "Mexico_Auto", "Mexico", 30000, 35, 0.4, 0.8
        )
        self.suppliers["Mexico_Electronics"] = SupplierDigitalTwin(
            "Mexico_Electronics", "Mexico", 25000, 40, 0.4, 0.82
        )
        self.suppliers["US_Premium"] = SupplierDigitalTwin(
            "US_Premium", "US", 25000, 50, 0.1, 0.95
        )
        self.suppliers["US_Tech"] = SupplierDigitalTwin(
            "US_Tech", "US", 20000, 55, 0.1, 0.97
        )

        self.logistics.ports = {
            "Shanghai": {"capacity": 1000000, "current_utilization": 0.7},
            "Los_Angeles": {"capacity": 800000, "current_utilization": 0.6},
            "New_York": {"capacity": 600000, "current_utilization": 0.5},
            "Rotterdam": {"capacity": 900000, "current_utilization": 0.65},
        }

        self.market.current_demand = {
            "Microprocessor": 10000,
            "Display": 8000,
            "Battery": 12000,
            "Sensor": 5000,
        }

    def update_system_state(self, time_step_hours=1):
        """Advance the model clock and update each simulated component."""
        self.current_time += timedelta(hours=time_step_hours)

        for supplier in self.suppliers.values():
            supplier.update_sensor_data(self.current_time)

        self.logistics.update_network_status(self.current_time)
        self.market.update_market_data(self.current_time)
        self.update_predictions()

    def update_predictions(self):
        """Set a simple disruption indicator from recent uptime readings."""
        for supplier in self.suppliers.values():
            recent_uptime = [
                value
                for _, value in supplier.sensor_data["operational_uptime"][-10:]
            ]
            if recent_uptime:
                average_uptime = np.mean(recent_uptime)
                if average_uptime < 0.95:
                    supplier.disruption_probability = 1.0 - average_uptime
                else:
                    supplier.disruption_probability = 0.01

    def optimize_sourcing_allocation(self):
        """Allocate demand proportionally to the suppliers' weighted scores."""
        demands = self.market.current_demand
        supplier_scores = {}

        for supplier_name, supplier in self.suppliers.items():
            adjusted_risk = supplier.risk_factor * (
                2 - supplier.operational_status
            )
            current_quality = (
                supplier.quality_score * supplier.quality_control_score
            )

            supplier_scores[supplier_name] = (
                0.6 * (1.0 / supplier.current_cost)
                + 0.3 * (1.0 / (1.0 + adjusted_risk))
                + 0.1 * current_quality
            )

        total_score = sum(supplier_scores.values())
        if total_score <= 0:
            raise ValueError("The total supplier score must be positive.")

        allocation = {}
        for product, demand in demands.items():
            allocation[product] = {
                supplier_name: demand * score / total_score
                for supplier_name, score in supplier_scores.items()
            }

        self.current_allocation = allocation
        return allocation

    def calculate_kpis(self, record_history=True):
        """Calculate procurement, risk, quality, delivery, and inventory KPIs."""
        if not self.current_allocation:
            return None

        total_cost = 0.0
        total_risk = 0.0
        total_quality = 0.0

        for product_allocation in self.current_allocation.values():
            for supplier_name, quantity in product_allocation.items():
                supplier = self.suppliers[supplier_name]
                total_cost += supplier.current_cost * quantity
                total_risk += supplier.risk_factor * quantity
                total_quality += supplier.quality_score * quantity

        delivery_performance = float(
            np.mean(
                [supplier.operational_status for supplier in self.suppliers.values()]
            )
        )

        recent_inventory_means = [
            np.mean([value for _, value in supplier.sensor_data["inventory_levels"][-5:]])
            for supplier in self.suppliers.values()
            if supplier.sensor_data["inventory_levels"]
        ]
        total_inventory = float(sum(recent_inventory_means))

        kpis = {
            "total_cost": total_cost,
            "supply_risk": total_risk,
            "quality_score": total_quality,
            "delivery_performance": delivery_performance,
            "inventory_levels": total_inventory,
        }

        if record_history:
            for metric, value in kpis.items():
                self.kpi_history[metric].append((self.current_time, value))

        return kpis


print("The integrated sourcing model is defined.")

## 3. Define the scenario engine

A **what-if scenario** changes one or more model inputs to examine how the modeled system responds. Scenario analysis is useful for tracing the consequences of an assumed event; it is not, by itself, a prediction that the event will occur.

The tariff scenario temporarily changes costs for selected suppliers, recalculates the allocation and KPIs, and then restores the original costs and allocation. Its cost impact is the scenario cost minus the baseline cost, using the same demand state for both calculations. The reported volume shifted away from affected suppliers is the decrease in their total assigned volume.

The port-closure and currency scenarios use explicit simplified arithmetic. They do not modify the sourcing allocation or represent a detailed route or financial-market model.

In [ ]:
class ScenarioEngine:
    """Evaluate simplified what-if scenarios for the integrated model."""

    def __init__(self):
        self.scenario_library = {
            "tariff_increase": self.simulate_tariff_increase,
            "port_closure": self.simulate_port_closure,
            "currency_devaluation": self.simulate_currency_devaluation,
            "supply_disruption": self.simulate_supply_disruption,
            "demand_surge": self.simulate_demand_surge,
            "quality_crisis": self.simulate_quality_crisis,
        }
        self.scenario_results = {}

    def simulate_tariff_increase(
        self, digital_twin, affected_suppliers, tariff_rate
    ):
        """Recalculate cost and allocation after a temporary tariff change."""
        unknown_suppliers = set(affected_suppliers) - set(digital_twin.suppliers)
        if unknown_suppliers:
            raise ValueError(f"Unknown suppliers: {sorted(unknown_suppliers)}")

        baseline_allocation = deepcopy(digital_twin.current_allocation)
        if not baseline_allocation:
            digital_twin.optimize_sourcing_allocation()
            baseline_allocation = deepcopy(digital_twin.current_allocation)

        baseline_kpis = digital_twin.calculate_kpis(record_history=False)
        original_costs = {
            name: digital_twin.suppliers[name].current_cost
            for name in affected_suppliers
        }

        try:
            for supplier_name in affected_suppliers:
                digital_twin.suppliers[supplier_name].apply_disruption(
                    "tariff_increase", tariff_rate
                )

            new_allocation = deepcopy(
                digital_twin.optimize_sourcing_allocation()
            )
            new_kpis = digital_twin.calculate_kpis(record_history=False)
            allocation_shift = self.analyze_allocation_shift(
                baseline_allocation, new_allocation, affected_suppliers
            )
        finally:
            for supplier_name, original_cost in original_costs.items():
                digital_twin.suppliers[supplier_name].current_cost = original_cost
            digital_twin.current_allocation = baseline_allocation

        return {
            "scenario_type": "tariff_increase",
            "cost_impact": new_kpis["total_cost"] - baseline_kpis["total_cost"],
            "allocation_shift": allocation_shift,
            "new_allocation": new_allocation,
            "new_kpis": new_kpis,
            "baseline_kpis": baseline_kpis,
        }

    def simulate_port_closure(self, digital_twin, closed_port, duration_days):
        """Estimate a simplified cost and delay impact for a port closure."""
        if duration_days < 0:
            raise ValueError("Duration must be non-negative.")
        if closed_port not in digital_twin.logistics.ports:
            raise ValueError(f"Unknown port: {closed_port}")

        logistics_impact = duration_days * 0.1
        rerouting_cost = 50000 * duration_days
        delivery_delay = duration_days * 0.5

        return {
            "scenario_type": "port_closure",
            "closed_port": closed_port,
            "duration_days": duration_days,
            "logistics_impact": logistics_impact,
            "rerouting_cost": rerouting_cost,
            "delivery_delay": delivery_delay,
            "customer_impact": max(0, 100 - delivery_delay * 10),
        }

    def simulate_currency_devaluation(
        self, digital_twin, currency, devaluation_rate
    ):
        """Estimate exposure and a stated proportional hedging benefit."""
        affected_suppliers = [
            name
            for name in digital_twin.suppliers
            if currency.lower() in name.lower()
        ]

        total_exposure = sum(
            digital_twin.suppliers[name].current_capacity
            * digital_twin.suppliers[name].current_cost
            for name in affected_suppliers
        )
        hedging_benefit = total_exposure * devaluation_rate * 0.8

        return {
            "scenario_type": "currency_devaluation",
            "total_exposure": total_exposure,
            "hedging_benefit": hedging_benefit,
            "net_impact": total_exposure * devaluation_rate - hedging_benefit,
            "affected_suppliers": affected_suppliers,
        }

    def simulate_supply_disruption(self, digital_twin, supplier_name, severity):
        """Describe a supplier disruption without changing the system state."""
        if supplier_name not in digital_twin.suppliers:
            raise ValueError(f"Unknown supplier: {supplier_name}")
        return {
            "scenario_type": "supply_disruption",
            "supplier_name": supplier_name,
            "severity": severity,
        }

    def simulate_demand_surge(self, digital_twin, product, surge_rate):
        """Describe a demand surge without changing the system state."""
        if product not in digital_twin.market.current_demand:
            raise ValueError(f"Unknown product: {product}")
        return {
            "scenario_type": "demand_surge",
            "product": product,
            "surge_rate": surge_rate,
            "current_demand": digital_twin.market.current_demand[product],
        }

    def simulate_quality_crisis(self, digital_twin, supplier_name, severity):
        """Describe a quality crisis without changing the system state."""
        if supplier_name not in digital_twin.suppliers:
            raise ValueError(f"Unknown supplier: {supplier_name}")
        return {
            "scenario_type": "quality_crisis",
            "supplier_name": supplier_name,
            "severity": severity,
        }

    def analyze_allocation_shift(
        self, baseline_allocation, new_allocation, affected_suppliers
    ):
        """Measure the reduction in volume assigned to affected suppliers."""
        baseline_volume = sum(
            quantity
            for product_allocation in baseline_allocation.values()
            for supplier_name, quantity in product_allocation.items()
            if supplier_name in affected_suppliers
        )
        new_volume = sum(
            quantity
            for product_allocation in new_allocation.values()
            for supplier_name, quantity in product_allocation.items()
            if supplier_name in affected_suppliers
        )
        return max(0.0, baseline_volume - new_volume)

    def run_scenario_batch(self, digital_twin, scenarios):
        """Run each configured scenario and return its model result."""
        results = []
        for scenario_config in scenarios:
            scenario_type = scenario_config["type"]
            params = scenario_config["params"]
            if scenario_type not in self.scenario_library:
                raise ValueError(f"Unknown scenario type: {scenario_type}")
            results.append(
                self.scenario_library[scenario_type](digital_twin, **params)
            )
        self.scenario_results = results
        return results


print("The scenario engine is defined.")

## 4. Initialize the example and calculate a baseline

The baseline is the model's starting point for comparisons. Creating the system and calculating its initial allocation before applying a scenario makes it possible to compare like with like: the tariff scenario uses the same demand state and compares its recalculated cost with this baseline.

In [ ]:
print("=== GLOBAL SOURCING DIGITAL-TWIN SIMULATION ===")

digital_twin = GlobalSourcingDigitalTwin()
digital_twin.initialize_suppliers()
digital_twin.scenario_engine = ScenarioEngine()

print(f"Suppliers initialized: {len(digital_twin.suppliers)}")
print(f"Logistics ports: {len(digital_twin.logistics.ports)}")
print(f"Market products: {len(digital_twin.market.current_demand)}")
print(f"Simulation time-scale setting: {digital_twin.simulation_speed}x")

initial_allocation = digital_twin.optimize_sourcing_allocation()
initial_kpis = digital_twin.calculate_kpis()

for product, demand in digital_twin.market.current_demand.items():
    allocated_quantity = sum(initial_allocation[product].values())
    assert np.isclose(allocated_quantity, demand, rtol=1e-10, atol=1e-8)

print("\nInitial KPIs:")
print(f"Total procurement cost: ${initial_kpis['total_cost']:,.2f}")
print(f"Supply-risk exposure: {initial_kpis['supply_risk']:,.2f}")
print(f"Quality-weighted quantity: {initial_kpis['quality_score']:,.2f}")
print(f"Mean supplier operational status: {initial_kpis['delivery_performance']:.2%}")
print(f"Estimated supplier inventory: {initial_kpis['inventory_levels']:,.2f}")

## 5. Evaluate what-if scenarios

The batch below evaluates three specified cases: a tariff increase for the two China-based suppliers, a closure of the Los Angeles port, and a currency devaluation associated with suppliers whose names contain `China`. These inputs define the scenarios; the resulting costs and impacts are calculated by the model when the notebook runs.

The tariff scenario recalculates allocation using the same score rule as the baseline. Its reported shift is the reduction in assigned volume for the affected suppliers, not a fixed percentage or a claim that volume moves to a particular country. Port-closure and currency results are scenario estimates under their stated simplifying assumptions.

In [ ]:
scenarios = [
    {
        "type": "tariff_increase",
        "params": {
            "affected_suppliers": [
                "China_Electronics",
                "China_Manufacturing",
            ],
            "tariff_rate": 0.25,
        },
    },
    {
        "type": "port_closure",
        "params": {"closed_port": "Los_Angeles", "duration_days": 14},
    },
    {
        "type": "currency_devaluation",
        "params": {"currency": "China", "devaluation_rate": 0.15},
    },
]

scenario_results = digital_twin.scenario_engine.run_scenario_batch(
    digital_twin, scenarios
)

print("=== SCENARIO RESULTS ===")
for result in scenario_results:
    print(f"\n{result['scenario_type'].replace('_', ' ').title()}:")

    if result["scenario_type"] == "tariff_increase":
        print(f"  Cost impact relative to baseline: ${result['cost_impact']:,.2f}")
        print(
            "  Volume shifted away from affected suppliers: "
            f"{result['allocation_shift']:,.2f} units"
        )
    elif result["scenario_type"] == "port_closure":
        print(f"  Rerouting-cost estimate: ${result['rerouting_cost']:,.2f}")
        print(f"  Estimated delivery delay: {result['delivery_delay']:.1f} days")
        print(
            "  Simplified customer-impact indicator: "
            f"{result['customer_impact']:.1f}"
        )
    elif result["scenario_type"] == "currency_devaluation":
        print(f"  Estimated exposure: ${result['total_exposure']:,.2f}")
        print(f"  Assumed hedging benefit: ${result['hedging_benefit']:,.2f}")
        print(f"  Estimated net impact: ${result['net_impact']:,.2f}")

print("\nThe scenario calculations are illustrative model outputs, not validated forecasts.")

## 6. Run the time-stepped simulation

A time-stepped simulation advances the model clock by a fixed interval and updates the components at each step. Here, each step represents six simulated hours. Sensor readings and demand are updated, a disruption may occur, and the score-based allocation is recalculated every four steps (one simulated day).

A disruption event changes one supplier's capacity, quality-control score, or operating status according to the selected event type. The simulation then records KPIs for the current allocation. Because the allocation rule does not enforce capacity limits, the recorded allocation should not be interpreted as a capacity-feasible operational plan.

In [ ]:
simulation_days = 30
hours_per_step = 6
total_steps = (simulation_days * 24) // hours_per_step

print(f"Simulation period: {simulation_days} days")
print(f"Number of time steps: {total_steps}")
print(f"Time represented by each step: {hours_per_step} hours")

simulation_kpis = []
optimization_updates = 0
disruption_events = 0

start_time = time.time()

for step in range(total_steps):
    digital_twin.update_system_state(hours_per_step)

    if np.random.random() < 0.05:
        disruption_events += 1
        supplier_name = np.random.choice(list(digital_twin.suppliers))
        disruption_type = np.random.choice(
            ["capacity_reduction", "quality_issue", "operational_issue"]
        )
        severity = np.random.uniform(0.1, 0.3)

        digital_twin.suppliers[supplier_name].apply_disruption(
            disruption_type, severity
        )
        digital_twin.kpi_history["disruption_events"].append(
            (digital_twin.current_time, supplier_name)
        )

    if step % 4 == 0:
        digital_twin.optimize_sourcing_allocation()
        optimization_updates += 1

    kpis = digital_twin.calculate_kpis()
    simulation_kpis.append(kpis)

simulation_time = time.time() - start_time

print("\n=== SIMULATION SUMMARY ===")
print(f"Steps completed: {len(simulation_kpis)}")
print(f"Allocation recalculations: {optimization_updates}")
print(f"Simulated supplier disruption events: {disruption_events}")
print(f"Python execution time: {simulation_time:.2f} seconds")

costs = [kpi["total_cost"] for kpi in simulation_kpis]
risks = [kpi["supply_risk"] for kpi in simulation_kpis]
deliveries = [kpi["delivery_performance"] for kpi in simulation_kpis]

print("\nKPIs across recorded time steps:")
print(f"Mean procurement cost: ${np.mean(costs):,.2f}")
print(f"Cost standard deviation: ${np.std(costs):,.2f}")
print(f"Mean supply-risk exposure: {np.mean(risks):,.2f}")
print(f"Mean supplier operational status: {np.mean(deliveries):.2%}")
print(f"Minimum supplier operational status: {np.min(deliveries):.2%}")

## 7. Visualize and interpret the simulation outputs

The plots below summarize quantities generated by the preceding cells. The KPI plot uses separate vertical axes because cost and operational status have different units. The supplier heatmap shows recent simulated uptime, quality, production relative to base capacity, and the model's disruption indicator. The scenario chart displays scenario-specific quantities, so its bars should not be compared as if they were all the same kind of economic measure.

A plot helps reveal patterns, but it does not validate the assumptions that generated those patterns. In particular, supplier production relative to base capacity is an observation from the simulation, not a constraint enforced by the sourcing allocation.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Global Sourcing Simulation: Model Outputs", fontsize=16)

steps = np.arange(1, len(simulation_kpis) + 1)
costs_in_thousands = np.array(costs) / 1000
delivery_percent = np.array(deliveries) * 100

ax_cost = axes[0, 0]
ax_delivery = ax_cost.twinx()
cost_line = ax_cost.plot(
    steps, costs_in_thousands, color="tab:blue", label="Procurement cost"
)[0]
delivery_line = ax_delivery.plot(
    steps,
    delivery_percent,
    color="tab:red",
    label="Mean operational status",
)[0]
ax_cost.set_xlabel("Simulation time step")
ax_cost.set_ylabel("Procurement cost (thousands of model currency units)")
ax_delivery.set_ylabel("Mean supplier operational status (%)")
ax_cost.set_title("Recorded KPIs over simulated time")
ax_cost.grid(True, alpha=0.3)
ax_cost.legend(
    [cost_line, delivery_line],
    [cost_line.get_label(), delivery_line.get_label()],
    loc="best",
)

supplier_names = list(digital_twin.suppliers)
performance_rows = []
for supplier_name in supplier_names:
    supplier = digital_twin.suppliers[supplier_name]
    recent_uptime = [
        value for _, value in supplier.sensor_data["operational_uptime"][-10:]
    ]
    recent_quality = [
        value for _, value in supplier.sensor_data["quality_metrics"][-10:]
    ]
    recent_production = [
        value for _, value in supplier.sensor_data["production_rate"][-10:]
    ]

    performance_rows.append(
        [
            np.mean(recent_uptime) * 100 if recent_uptime else np.nan,
            np.mean(recent_quality) * 100 if recent_quality else np.nan,
            (
                np.mean(recent_production) / supplier.base_capacity * 100
                if recent_production
                else np.nan
            ),
            (1 - supplier.disruption_probability) * 100,
        ]
    )

performance_df = pd.DataFrame(
    performance_rows,
    index=supplier_names,
    columns=["Uptime (%)", "Quality (%)", "Production / base capacity (%)", "Reliability indicator (%)"],
)
sns.heatmap(
    performance_df,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn",
    vmin=0,
    vmax=100,
    ax=axes[0, 1],
)
axes[0, 1].set_title("Recent supplier simulation indicators")
axes[0, 1].set_xlabel("Indicator")
axes[0, 1].set_ylabel("Supplier")

scenario_labels = []
scenario_values = []
for result in scenario_results:
    scenario_labels.append(result["scenario_type"].replace("_", " ").title())
    if result["scenario_type"] == "tariff_increase":
        scenario_values.append(result["cost_impact"])
    elif result["scenario_type"] == "port_closure":
        scenario_values.append(result["rerouting_cost"])
    elif result["scenario_type"] == "currency_devaluation":
        scenario_values.append(result["net_impact"])

scenario_colors = [
    "tab:red" if value >= 0 else "tab:green" for value in scenario_values
]
axes[1, 0].bar(scenario_labels, scenario_values, color=scenario_colors, alpha=0.8)
axes[1, 0].axhline(0, color="black", linewidth=0.8)
axes[1, 0].set_title("Scenario-specific cost or impact measure")
axes[1, 0].set_xlabel("Scenario")
axes[1, 0].set_ylabel("Amount (model currency units)")
axes[1, 0].tick_params(axis="x", rotation=25)
axes[1, 0].grid(True, alpha=0.3, axis="y")

final_allocation = digital_twin.current_allocation
product_totals = {
    product: sum(allocations.values())
    for product, allocations in final_allocation.items()
}
axes[1, 1].bar(
    list(product_totals),
    list(product_totals.values()),
    color="tab:purple",
    alpha=0.8,
)
axes[1, 1].set_title("Final allocated quantity by product")
axes[1, 1].set_xlabel("Product")
axes[1, 1].set_ylabel("Allocated quantity (units)")
axes[1, 1].tick_params(axis="x", rotation=25)
axes[1, 1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

allocation_by_supplier = {
    supplier_name: sum(
        product_allocation.get(supplier_name, 0.0)
        for product_allocation in final_allocation.values()
    )
    for supplier_name in supplier_names
}

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(
    list(allocation_by_supplier),
    list(allocation_by_supplier.values()),
    color="tab:blue",
    alpha=0.8,
)
ax.set_title("Final heuristic allocation by supplier")
ax.set_xlabel("Supplier")
ax.set_ylabel("Allocated quantity (units)")
ax.tick_params(axis="x", rotation=35)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 8. Summarize the model run

The summary reports values calculated or recorded by this notebook run. It does not present external performance claims as measured results. The interpretation should remain within the model's scope: these are outputs of a small stochastic simulation and a proportional score-based allocation rule, with the capacity and integration limitations stated above.

In practice, this style of model can support structured discussion about which assumptions need better data, which disruptions deserve analysis, and which system components should be integrated. Before using such outputs for operational decisions, a real application would require validated data, explicit feasibility constraints, suitable product-supplier relationships, and testing against observed outcomes.

In [ ]:
print("=== DIGITAL-TWIN SIMULATION SUMMARY ===")
print(f"Simulated days: {simulation_days}")
print(f"Time steps recorded: {len(simulation_kpis)}")
print(f"Allocation recalculations: {optimization_updates}")
print(f"Disruption events generated: {disruption_events}")
print(f"Mean recorded procurement cost: ${np.mean(costs):,.2f}")
print(f"Mean recorded supply-risk exposure: {np.mean(risks):,.2f}")
print(f"Mean recorded supplier operational status: {np.mean(deliveries):.2%}")

tariff_result = next(
    result
    for result in scenario_results
    if result["scenario_type"] == "tariff_increase"
)
print(
    "Tariff scenario cost impact relative to its baseline: "
    f"${tariff_result['cost_impact']:,.2f}"
)

print("\nInterpretation:")
print(
    "The allocation rule distributes each product's demand according to "
    "supplier scores; it does not enforce capacity limits."
)
print(
    "Scenario and simulation values are conditional on the stated model "
    "assumptions and should not be treated as validated forecasts."
)
print(
    "The model demonstrates an integrated simulation workflow, not a "
    "production-ready digital-twin system."
)